# Notebook 02 — Estadística descriptiva

---

### Entorno y persistencia de resultados

La celda siguiente fija tres cosas que condicionan la reproducibilidad del experimento:

**Semilla fija.** `RANDOM_STATE = 42` se aplica a la partición train/test y al ajuste de todos los
modelos. Sin ella, cada ejecución produciría particiones distintas y las métricas no serían
comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Los resultados se escriben en `MyDrive/hotel_booking` y no en el
disco temporal de Colab, que se borra al cerrar la sesión. Esto permite que el experimento se ejecute
en varias sesiones sin repetir etapas: el notebook 03 deja las particiones preparadas y los notebooks
04 y 05 las consumen tal cual, garantizando que todos operan exactamente sobre los mismos datos.
Si el montaje no se completa, la celda interrumpe la ejecución en lugar de escribir en una ubicación
volátil.

**Registro de las figuras.** La función `guardar` escribe cada gráfico en `splits/` como PNG a 200
dpi. Se invoca siempre antes de `plt.show()`, porque mostrar la figura vacía el buffer de matplotlib
y el archivo resultante quedaría en blanco.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = "/content/drive/MyDrive/hotel_booking"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath("./hotel_booking")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)

def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    print("Grafico guardado:", destino)

print("Guardando en Google Drive" if EN_DRIVE else "Google Drive no disponible: guardando local")
print("Carpeta de trabajo:", RUTA)
print("Graficos (splits) :", CARPETA_SPLITS)

In [ ]:
URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(URL)
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])

## 2.2 Estadística descriptiva (7 %)

### Estadísticos globales de las variables numéricas

In [ ]:
num_cols = [
    "lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes",
    "days_in_waiting_list", "adr", "required_car_parking_spaces",
    "total_of_special_requests",
]

resumen = df[num_cols].describe().T
resumen["rango"] = resumen["max"] - resumen["min"]
resumen.round(2)

Las escalas son muy dispares: `lead_time` llega a 737 días y `adr` supera los 5.000 euros, mientras
que `babies` no pasa de unas pocas unidades. Esto justifica el escalado que se aplica en el
notebook 03.

### Estadísticos desagregados por clase

Es la parte que permite anticipar qué variables separan los grupos.

In [ ]:
vars_clave = ["lead_time", "adr", "total_of_special_requests",
              "previous_cancellations", "booking_changes", "required_car_parking_spaces"]

por_clase = df.groupby("reservation_status")[vars_clave].agg(["mean", "median", "std"]).round(2)
por_clase

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, v in zip(axes, ["lead_time", "adr", "total_of_special_requests"]):
    sns.boxplot(data=df, x="reservation_status", y=v, ax=ax, showfliers=False)
    ax.set_title(f"{v} por clase")
    ax.set_xlabel("")
plt.tight_layout()
guardar("02_boxplots_por_clase")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(data=df, x="lead_time", hue="reservation_status",
             element="step", stat="density", common_norm=False, bins=50, ax=axes[0])
axes[0].set_title("Distribucion de lead_time por clase")

tabla_dep = pd.crosstab(df["deposit_type"], df["reservation_status"], normalize="index") * 100
tabla_dep.plot(kind="barh", stacked=True, ax=axes[1])
axes[1].set_title("Composicion de clases segun tipo de deposito (%)")
axes[1].set_xlabel("% de reservas")
plt.tight_layout()
guardar("02_leadtime_y_tipo_deposito")
plt.show()

print(tabla_dep.round(1))

### Interpretación y vínculo con la separabilidad

Redactá esta parte con los números que arroje tu ejecución. Estas son las tres lecturas a contrastar:

- `lead_time` es mucho mayor en `Canceled`: reservar con mucha anticipación da más tiempo y más
  motivos para cancelar. La diferencia entre medianas es grande y va en la dirección esperada.
- `required_car_parking_spaces` y `total_of_special_requests` son mayores en `Check-Out`: quien pide
  estacionamiento o servicios extra ya organizó el viaje y es menos probable que falle.
- **El hallazgo menos obvio está en `No-Show`.** No es un punto intermedio entre las otras dos:
  tiene la anticipación **más corta de las tres** y un historial de cancelaciones tan limpio como el
  de `Check-Out`. Es decir, **al momento de reservar una futura no-presentación se parece a una
  reserva normal**, no a una cancelación. Anotá esto: explica el principal error del modelo, que se
  analiza en el notebook 05.

Prestá atención especial a la tabla de `deposit_type`: la proporción de cancelaciones entre las
reservas con depósito no reembolsable es llamativamente alta. Es un hallazgo contraintuitivo que
conviene tener anotado desde ya, porque reaparece en los coeficientes del modelo.

---

**Siguiente paso:** `03_calidad_desbalance_y_escalado.ipynb`.